#Phase 1-2

In [43]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [44]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [45]:
pip install fastf1

Note: you may need to restart the kernel to use updated packages.


In [46]:
import pandas as pd
import numpy as np
import fastf1


fastf1.Cache.enable_cache('fastf1_cache') 
pd.options.mode.chained_assignment = None

In [47]:
# Load a specific session (e.g., 2023 Bahrain Grand Prix - Race)
year, grand_prix, session_type = 2023, 'Bahrain', 'R'
session = fastf1.get_session(year, grand_prix, session_type)
session.load(telemetry=True, laps=True, weather=True)

df = session.laps

# Tag the Race_ID for Group K-Fold Cross Validation later
df['Race_ID'] = f"{year}_{grand_prix}"

# Define the Finishing_Tier dependent variable based on final positions
df['Finishing_Tier'] = np.select(
    [
        df['Position'] <= 3,
        (df['Position'] > 3) & (df['Position'] <= 10)
    ], 
    ['Podium', 'Point Scorer'], 
    default='No Points'
)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [48]:
df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Race_ID,Finishing_Tier
0,0 days 01:04:15.902000,VER,1,0 days 00:01:39.019000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:42.414000,...,0 days 01:02:36.652000,2023-03-05 15:03:38.501,12,1.0,False,,False,False,2023_Bahrain,Podium
1,0 days 01:05:53.876000,VER,1,0 days 00:01:37.974000,2.0,1.0,NaT,NaT,0 days 00:00:31.342000,0 days 00:00:42.504000,...,0 days 01:04:15.902000,2023-03-05 15:05:17.751,12,1.0,False,,False,True,2023_Bahrain,Podium
2,0 days 01:07:31.882000,VER,1,0 days 00:01:38.006000,3.0,1.0,NaT,NaT,0 days 00:00:31.388000,0 days 00:00:42.469000,...,0 days 01:05:53.876000,2023-03-05 15:06:55.725,1,1.0,False,,False,True,2023_Bahrain,Podium
3,0 days 01:09:09.858000,VER,1,0 days 00:01:37.976000,4.0,1.0,NaT,NaT,0 days 00:00:31.271000,0 days 00:00:42.642000,...,0 days 01:07:31.882000,2023-03-05 15:08:33.731,1,1.0,False,,False,True,2023_Bahrain,Podium
4,0 days 01:10:47.893000,VER,1,0 days 00:01:38.035000,5.0,1.0,NaT,NaT,0 days 00:00:31.244000,0 days 00:00:42.724000,...,0 days 01:09:09.858000,2023-03-05 15:10:11.707,1,1.0,False,,False,True,2023_Bahrain,Podium


In [49]:
# --- Merge final race result/status into laps df ---
results = session.results[['Abbreviation', 'Status', 'ClassifiedPosition']]
# Merge into laps df on driver abbreviation
df = session.laps.copy()
df = df.merge(results, left_on='Driver', right_on='Abbreviation', how='left')
# --- Identify DNFs ---
# ClassifiedPosition is NaN or 'NC' for drivers who did not finish
# True DNFs: only 'Retired' status
dnf_mask = (df['Status'] == 'Retired') | (df['ClassifiedPosition'] == 'R')

df.loc[dnf_mask, 'Finishing_Tier'] = 'No Points'

# --- Truncate telemetry: drop laps with no recorded lap time (e.g., lap of DNF) ---
df = df.dropna(subset=['LapTime'])

# --- Forward-fill remaining gaps (e.g., sensor dropouts on valid laps) ---
df.ffill(inplace=True)


C:\Users\loren\AppData\Local\Temp\ipykernel_3712\4195099253.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.ffill(inplace=True)


In [50]:
#DNF Protocol
dnf_mask = (df['Status'] == 'Retired') | (df['ClassifiedPosition'] == 'R')


df.loc[dnf_mask, 'Finishing_Tier'] = 'No Points'


def assign_tier(row):
    if row['Finishing_Tier'] == 'No Points': 
        return 'No Points'
    try:
        pos = int(row['ClassifiedPosition'])
        if pos <= 3:
            return 'Podium'
        elif pos <= 10:
            return 'Point Scorer'
        else:
            return 'No Points'
    except (ValueError, TypeError):
        return 'No Points'

df['Finishing_Tier'] = df.apply(assign_tier, axis=1)
df = df.dropna(subset=['LapTime'])
df.ffill(inplace=True)

# Sanity check
print(df.groupby('Driver')['Finishing_Tier'].first().sort_values())


Driver
ALB       No Points
TSU       No Points
SAR       No Points
PIA       No Points
OCO       No Points
MAG       No Points
LEC       No Points
NOR       No Points
GAS       No Points
DEV       No Points
HUL       No Points
ZHO       No Points
VER          Podium
PER          Podium
ALO          Podium
HAM    Point Scorer
RUS    Point Scorer
SAI    Point Scorer
BOT    Point Scorer
STR    Point Scorer
Name: Finishing_Tier, dtype: object


## Categorical Encoding (k-1 Dummies)

In [51]:
# Convert tire compounds (Soft, Medium, Hard, Intermediate, Wet) into k-1 dummy variables
df = pd.get_dummies(df, columns=['Compound'], prefix='Tire', drop_first=True)

#(0 or 1)
for col in df.columns:
    if col.startswith('Tire_'):
        df[col] = df[col].astype(int)

##Feature Engineering and Mean-Center Itneraction

In [52]:
# 1. Engineer a proxy for Fuel Mass if exact weight isn't available
# Assuming a standard 110kg starting load burning off linearly over total race laps
total_laps = session.total_laps
df['Fuel_Mass_Proxy'] = 110 - (110 / total_laps) * df['LapNumber']

# 2. Extract Tire Degradation (using TireLife as a basic proxy if actual degradation slope isn't pre-calculated)
# In a full model, this would be the derivative of lap time over a stint.
df['Tire_Age'] = df['TyreLife']

# 3. Create the Mean-Centered Dynamic Interaction Term
mean_tire_age = df['Tire_Age'].mean()
mean_fuel_mass = df['Fuel_Mass_Proxy'].mean()

df['Degradation_x_Fuel'] = (df['Tire_Age'] - mean_tire_age) * (df['Fuel_Mass_Proxy'] - mean_fuel_mass)

In [54]:
df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Race_ID,Finishing_Tier,Abbreviation,Status,ClassifiedPosition,Tire_MEDIUM,Tire_SOFT,Fuel_Mass_Proxy,Tire_Age,Degradation_x_Fuel
0,0 days 01:04:15.902000,VER,1,0 days 00:01:39.019000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:42.414000,...,2023_Bahrain,Podium,VER,Finished,1,0,1,108.070175,4.0,-301.785428
1,0 days 01:05:53.876000,VER,1,0 days 00:01:37.974000,2.0,1.0,NaT,NaT,0 days 00:00:31.342000,0 days 00:00:42.504000,...,2023_Bahrain,Podium,VER,Finished,1,0,1,106.140351,5.0,-240.583466
2,0 days 01:07:31.882000,VER,1,0 days 00:01:38.006000,3.0,1.0,NaT,NaT,0 days 00:00:31.388000,0 days 00:00:42.469000,...,2023_Bahrain,Podium,VER,Finished,1,0,1,104.210526,6.0,-183.241153
3,0 days 01:09:09.858000,VER,1,0 days 00:01:37.976000,4.0,1.0,NaT,NaT,0 days 00:00:31.271000,0 days 00:00:42.642000,...,2023_Bahrain,Podium,VER,Finished,1,0,1,102.280702,7.0,-129.758489
4,0 days 01:10:47.893000,VER,1,0 days 00:01:38.035000,5.0,1.0,NaT,NaT,0 days 00:00:31.244000,0 days 00:00:42.724000,...,2023_Bahrain,Podium,VER,Finished,1,0,1,100.350877,8.0,-80.135474


In [ ]:
#Handoff
# Select final columns for the model
# Include target, Race_ID, main effects, interaction terms, and k-1 dummies
features_to_export = [
    'Race_ID', 'Driver', 'LapNumber', 'Finishing_Tier', 
    'Tire_Age', 'Fuel_Mass_Proxy', 'Degradation_x_Fuel'
] + [col for col in df.columns if col.startswith('Tire_')]

df_clean = df[features_to_export]
df_clean.to_csv('f1_clean_data.csv', index=False)
print("Data Prep Complete, Extracted")

Data Prep Complete, Extracted
